# Solution: Introduction to Linear Regression

**Course context:** Codecademy-style *Introduction to Linear Regression* (from-scratch gradient descent + scikit-learn).

**Goal:** Fit `weight ≈ m · height + b` for 200 professional baseball players. Implement gradient descent yourself, then verify with scikit-learn and the closed-form solution.

**Data:** `data/heights.csv`.

---

## Project Flowchart

![Flowchart](introduction_to_linear_regression_flowchart.png)

### Audience adaptation reminder
- Technical readers → gradients, residual diagnostics, method comparison.
- Domain experts (sports science) → predictive usefulness of height for weight; limitations.
- Executives / non-specialists → headline slope + R² in plain language; one clean chart.


## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load & Explore the Data

In [ ]:
df = pd.read_csv('data/heights.csv')
print('Shape:', df.shape)
display(df.head())
display(df.describe().round(2))

plt.figure(figsize=(7,5))
plt.scatter(df['height'], df['weight'], alpha=0.65, edgecolor='k', linewidth=0.3)
plt.xlabel('Height (inches)')
plt.ylabel('Weight (lb)')
plt.title('Baseball Players: Height vs Weight (n=200)')
plt.tight_layout()
plt.show()

## 2. Points and Lines — Manual Prediction (lemonade warm-up)

In [ ]:
months = list(range(1, 13))
revenue = [52, 74, 79, 95, 115, 110, 129, 126, 147, 146, 156, 184]

m, b = 11, 45          # reasonable eyeball values
y_pred = [m * x + b for x in months]

plt.figure(figsize=(7,4))
plt.plot(months, revenue, 'o', label='Observed revenue')
plt.plot(months, y_pred, '-', label=f'Manual line (m={m}, b={b})')
plt.xlabel('Month'); plt.ylabel('Revenue ($)')
plt.title("Sandra's Lemonade Stand — Manual Line")
plt.legend(); plt.tight_layout(); plt.show()

print('Approximate month-13 revenue:', m*13 + b)

## 3. Loss — Sum of Squared Errors

In [ ]:
x = [1, 2, 3]
y = [5, 1, 3]

m1, b1 = 1, 0
m2, b2 = 0.5, 1

y_pred1 = [m1 * xi + b1 for xi in x]
y_pred2 = [m2 * xi + b2 for xi in x]

total_loss1 = 0
total_loss2 = 0
for i in range(len(y)):
    total_loss1 += (y[i] - y_pred1[i]) ** 2
    total_loss2 += (y[i] - y_pred2[i]) ** 2

print('Loss line 1:', total_loss1)
print('Loss line 2:', total_loss2)
better_fit = 1 if total_loss1 < total_loss2 else 2
print('Better fit: line', better_fit)

# Alternate (vectorised)
print('Vectorised losses:',
      np.sum((np.array(y) - np.array(y_pred1))**2),
      np.sum((np.array(y) - np.array(y_pred2))**2))

## 4–5. Gradients for Intercept and Slope

In [ ]:
def get_gradient_at_b(x, y, m, b):
    N = len(x)
    diff = 0.0
    for i in range(N):
        diff += y[i] - (m * x[i] + b)
    return -2.0 / N * diff

def get_gradient_at_m(x, y, m, b):
    N = len(x)
    diff = 0.0
    for i in range(N):
        diff += x[i] * (y[i] - (m * x[i] + b))
    return -2.0 / N * diff

# Vectorised alternates
def get_gradient_at_b_vec(x, y, m, b):
    return -2.0 * np.mean(y - (m * x + b))

def get_gradient_at_m_vec(x, y, m, b):
    return -2.0 * np.mean(x * (y - (m * x + b)))

print('Loop  gb:', get_gradient_at_b([1,2,3],[5,1,3],0.5,1))
print('Vec   gb:', get_gradient_at_b_vec(np.array([1.,2,3]), np.array([5.,1,3]), 0.5, 1))
print('Loop  gm:', get_gradient_at_m([1,2,3],[5,1,3],0.5,1))
print('Vec   gm:', get_gradient_at_m_vec(np.array([1.,2,3]), np.array([5.,1,3]), 0.5, 1))

## 6. One Gradient Step

In [ ]:
def step_gradient(x, y, b_current, m_current, learning_rate):
    b_grad = get_gradient_at_b(x, y, m_current, b_current)
    m_grad = get_gradient_at_m(x, y, m_current, b_current)
    b = b_current - learning_rate * b_grad
    m = m_current - learning_rate * m_grad
    return [b, m]

print('One step from (0,0) on lemonade:',
      step_gradient(months, revenue, 0, 0, 0.01))

## 7. Full Gradient Descent Loop

In [ ]:
def gradient_descent(x, y, learning_rate, num_iterations, return_history=False):
    b, m = 0.0, 0.0
    history = []
    for _ in range(num_iterations):
        b, m = step_gradient(x, y, b, m, learning_rate)
        if return_history:
            yhat = [m * xi + b for xi in x]
            loss = sum((yi - yh)**2 for yi, yh in zip(y, yhat)) / len(y)
            history.append((b, m, loss))
    if return_history:
        return [b, m], history
    return [b, m]

# Lemonade demonstration
b_lem, m_lem = gradient_descent(months, revenue, learning_rate=0.01, num_iterations=1000)
print(f'Lemonade GD  →  m={m_lem:.3f}, b={b_lem:.3f}')

y_lem = [m_lem * x + b_lem for x in months]
plt.figure(figsize=(7,4))
plt.plot(months, revenue, 'o', label='Data')
plt.plot(months, y_lem, '-', label='GD fit')
plt.xlabel('Month'); plt.ylabel('Revenue'); plt.legend(); plt.title('Lemonade — Gradient Descent');
plt.tight_layout(); plt.show()

## 8. From-Scratch GD on Height → Weight

Raw heights are large; without centering a very small learning rate is required and convergence from zero is slow. We therefore show both approaches.

In [ ]:
X = df['height'].values.astype(float)
y = df['weight'].values.astype(float)

# --- Centered version (recommended) ---
Xc = X - X.mean()
yc = y - y.mean()

(b_c, m_c), hist = gradient_descent(Xc.tolist(), yc.tolist(),
                                    learning_rate=0.001,
                                    num_iterations=2000,
                                    return_history=True)
m_gd = m_c
b_gd = y.mean() - m_gd * X.mean()   # recover original intercept
print(f'Centered GD  →  m={m_gd:.4f}, b={b_gd:.4f}')

yhat_gd = m_gd * X + b_gd
mse_gd = np.mean((y - yhat_gd)**2)
print(f'MSE (GD) = {mse_gd:.2f}')

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.6, edgecolor='k', linewidth=0.3, label='Data')
xx = np.linspace(X.min()-0.5, X.max()+0.5, 100)
plt.plot(xx, m_gd*xx + b_gd, 'r-', lw=2, label=f'GD: y={m_gd:.2f}x+{b_gd:.1f}')
plt.xlabel('Height (in)'); plt.ylabel('Weight (lb)')
plt.title('From-Scratch Gradient Descent Fit')
plt.legend(); plt.tight_layout(); plt.show()

# Loss trajectory
losses = [h[2] for h in hist]
plt.figure(figsize=(6,3.5))
plt.plot(losses)
plt.xlabel('Iteration'); plt.ylabel('MSE (centered data)')
plt.title('Convergence of Centered GD'); plt.yscale('log')
plt.tight_layout(); plt.show()

## 9. scikit-learn LinearRegression

In [ ]:
X_2d = X.reshape(-1, 1)
model = LinearRegression().fit(X_2d, y)
m_sk = model.coef_[0]
b_sk = model.intercept_
r2_sk = model.score(X_2d, y)
print(f'sklearn     →  m={m_sk:.4f}, b={b_sk:.4f}, R²={r2_sk:.4f}')

yhat_sk = model.predict(X_2d)

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.6, edgecolor='k', linewidth=0.3, label='Data')
plt.plot(xx, m_sk*xx + b_sk, 'g-', lw=2, label=f'sklearn: y={m_sk:.2f}x+{b_sk:.1f}')
plt.xlabel('Height (in)'); plt.ylabel('Weight (lb)')
plt.title('scikit-learn LinearRegression Fit')
plt.legend(); plt.tight_layout(); plt.show()

## 10. More Practice — Closed Form + Residuals + Method Comparison

In [ ]:
# Closed-form (normal equations for simple LR)
x_bar, y_bar = X.mean(), y.mean()
m_cf = np.sum((X - x_bar) * (y - y_bar)) / np.sum((X - x_bar)**2)
b_cf = y_bar - m_cf * x_bar
print(f'Closed-form →  m={m_cf:.4f}, b={b_cf:.4f}')

# Comparison table
comparison = pd.DataFrame({
    'Method': ['Centered GD', 'sklearn', 'Closed-form'],
    'Slope m': [m_gd, m_sk, m_cf],
    'Intercept b': [b_gd, b_sk, b_cf]
})
display(comparison.round(4))

# Residual diagnostics
resid = y - yhat_sk
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(yhat_sk, resid, alpha=0.6, edgecolor='k', linewidth=0.3)
axes[0].axhline(0, color='r', ls='--')
axes[0].set_xlabel('Fitted weight'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')
axes[1].hist(resid, bins=20, edgecolor='k', alpha=0.7)
axes[1].set_xlabel('Residual'); axes[1].set_title('Residual Distribution')
plt.suptitle(f'Residual Analysis (R² = {r2_sk:.3f})')
plt.tight_layout(); plt.show()

print('Residual mean (should be ≈0):', resid.mean().round(6))
print('Residual std:', resid.std().round(2))

## 11. Simulation Section — Learning Rate, Noise & Sample Size

Modify the parameters below and re-run the cell to explore sensitivity.

In [ ]:
# === TUNABLE PARAMETERS ===
LEARNING_RATE = 0.001
N_ITER        = 2000
NOISE_STD     = 0.0      # try 5, 10, 20
SAMPLE_FRAC   = 1.0      # try 0.25, 0.5, 0.8
N_REPS        = 30       # Monte-Carlo repetitions when noise or sub-sampling is active
# ===========================

def run_one_sim(X, y, alpha, n_iter, noise_std, sample_frac):
    n = len(X)
    idx = np.random.choice(n, size=max(10, int(n * sample_frac)), replace=False)
    Xs, ys = X[idx], y[idx].copy()
    if noise_std > 0:
        ys = ys + np.random.normal(0, noise_std, size=len(ys))
    Xc = Xs - Xs.mean()
    yc = ys - ys.mean()
    b_c, m_c = gradient_descent(Xc.tolist(), yc.tolist(), alpha, n_iter)
    m = m_c
    b = ys.mean() - m * Xs.mean()
    yhat = m * Xs + b
    ss_res = np.sum((ys - yhat)**2)
    ss_tot = np.sum((ys - ys.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return m, b, r2

ms, bs, r2s = [], [], []
for _ in range(N_REPS):
    m, b, r2 = run_one_sim(X, y, LEARNING_RATE, N_ITER, NOISE_STD, SAMPLE_FRAC)
    ms.append(m); bs.append(b); r2s.append(r2)

print(f'Settings: α={LEARNING_RATE}, iters={N_ITER}, noise={NOISE_STD}, frac={SAMPLE_FRAC}, reps={N_REPS}')
print(f'Slope   mean±std : {np.mean(ms):.4f} ± {np.std(ms):.4f}')
print(f'Intercept mean±std: {np.mean(bs):.2f} ± {np.std(bs):.2f}')
print(f'R²      mean±std : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].hist(ms, bins=12, edgecolor='k', alpha=0.75); axes[0].axvline(m_sk, color='r', ls='--', label='sklearn')
axes[0].set_title('Slope distribution'); axes[0].legend(fontsize=8)
axes[1].hist(bs, bins=12, edgecolor='k', alpha=0.75); axes[1].axvline(b_sk, color='r', ls='--')
axes[1].set_title('Intercept distribution')
axes[2].hist(r2s, bins=12, edgecolor='k', alpha=0.75); axes[2].axvline(r2_sk, color='r', ls='--')
axes[2].set_title('R² distribution')
plt.suptitle('Monte-Carlo Sensitivity Simulation')
plt.tight_layout(); plt.show()

### Extra simulation: effect of learning rate on loss trajectory

In [ ]:
def loss_trajectory(X, y, alpha, n_iter=1500):
    Xc, yc = X - X.mean(), y - y.mean()
    b, m = 0.0, 0.0
    losses = []
    for _ in range(n_iter):
        b, m = step_gradient(Xc.tolist(), yc.tolist(), b, m, alpha)
        yhat = m * Xc + b
        losses.append(np.mean((yc - yhat)**2))
    return losses

plt.figure(figsize=(8,4))
for alpha in [1e-4, 5e-4, 0.001, 0.005]:
    losses = loss_trajectory(X, y, alpha)
    plt.plot(losses, label=f'α={alpha}')
plt.yscale('log'); plt.xlabel('Iteration'); plt.ylabel('MSE (centered)')
plt.title('Learning-Rate Effect on Convergence Speed')
plt.legend(); plt.tight_layout(); plt.show()

## Cheat Sheet

| Concept | Formula / Code |
|---------|----------------|
| Line | `yhat = m*x + b` |
| SSE / Loss | `sum( (y - yhat)**2 )` or `np.mean((y-yhat)**2)` for MSE |
| Gradient b | `-2/N * sum(y - (m*x+b))` |
| Gradient m | `-2/N * sum(x * (y - (m*x+b)))` |
| Update | `param -= learning_rate * gradient` |
| Convergence | parameters (or loss) change very little between iterations |
| Learning rate too small | painfully slow convergence |
| Learning rate too large | divergence or oscillation |
| sklearn | `LinearRegression().fit(X, y)` → `.coef_`, `.intercept_`, `.predict`, `.score` |
| Closed form | `m = cov(x,y)/var(x)`, `b = mean(y)-m*mean(x)` |
| Centering trick | GD on centered data; `b_orig = y.mean() - m * X.mean()` |

**Audience quick tips**
- Analysts / technical supervisors → residual plot + R² + method comparison table.
- Executives → “Each extra inch of height is associated with roughly **3.43 lb** higher weight; the linear model explains about **31 %** of the variance in weight.”
- Non-specialists → clean scatter with one fitted line; no mention of gradients or loss functions.

**Key results on this dataset**
- Slope ≈ **3.43 lb per inch**
- Intercept ≈ **-106 lb** (not directly interpretable far outside the observed height range)
- R² ≈ **0.31** (height alone explains a moderate fraction of weight variation)
